### Notebook scope

### Purpose
Document available hindcast cases and product coverage.

### Scientific question
Which cases can support source diagnosis or INT-vs-NOCOUPL feedback tests?

### Method
Scan /mnt/soclim0/public_data/weiji/Hindcast and parse year, init month, configuration, and perturbation suffix.

### Expected output
One inventory CSV and one availability matrix figure.

### Interpretation
Rows or cells marked missing should be treated as not applicable rather than failed analysis.

### Caveat
0003 is a special branch and is not forced into a NOCOUPL pair.


### Setup imports and shared paths

### Purpose
Load the shared Extention_analysis utility module and create output directories.

### Scientific question
Can every notebook start from a clean kernel and still find the same data roots and output roots?

### Method
Use DATA_ROOT, HINDCAST_ROOT, BWCN_ROOT, B2000_ROOT, WORK_ROOT, FIG_DIR, TAB_DIR, CACHE_DIR, and LOG_DIR from hindcast_extension_utils.py. No diagnostics are calculated in this cell.

### Expected output
Printed path check only. No figure is generated by this setup cell.

### Interpretation
Successful execution means the notebook can access the common utilities and write outputs in the agreed directory tree.

### Caveat
This setup does not prove that every downstream data product exists; missing products are logged later.


In [ ]:
from pathlib import Path
import sys

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

import hindcast_extension_utils as hx
from hindcast_extension_utils import *
_assign_member_short = hx._assign_member_short

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)

### Build case inventory and availability matrix

### Purpose
Create the master case inventory and matrix figure.

### Scientific question
Which years/init months/configurations exist, and which have O3, EPFlux, U, Z3, FWD, and AO/NAM products?

### Method
Rows are selected years 0003, 0008, 0013, 0014, 0019. Columns are init months Jan, Feb, Mar. Cell text lists available experiment types and member counts.

### Expected output
outputs/tables/00_case_inventory.csv and outputs/figures/00_hindcast_availability_matrix.png/pdf.

### Interpretation
A cell with INT/NOCOUPL entries indicates a possible feedback comparison. A gray empty cell means no case was found.

### Caveat
Inventory reflects cleaned products currently present. OMEGA gaps are not treated as fatal because EP100 uses the no-omega-consistent hindcast EPflux_daily_ubar product.


In [ ]:
inv = discover_hindcast_cases(HINDCAST_ROOT)
inv_path = TAB_DIR / "00_case_inventory.csv"
inv.to_csv(inv_path, index=False)
print(inv[["case", "year", "init_month", "config", "perturbation", "n_members", "can_source_diagnose"]].to_string(index=False))

years = SELECTED_YEARS
months = ["01", "02", "03"]
month_labels = {"01": "Jan", "02": "Feb", "03": "Mar"}
fig, ax = plt.subplots(figsize=(11.5, 5.6), constrained_layout=True)
ax.set_xlim(0, len(months)); ax.set_ylim(0, len(years))
ax.set_xticks(np.arange(len(months)) + 0.5, [month_labels[m] for m in months])
ax.set_yticks(np.arange(len(years)) + 0.5, years)
ax.invert_yaxis()
for yi, year in enumerate(years):
    for mi, month in enumerate(months):
        sub = inv[(inv["year"] == year) & (inv["init_month"] == month)]
        color = "#f0f0f0" if sub.empty else "#d9f0d3"
        if (not sub.empty) and any(sub["config"] == "NOCOUPL"):
            color = "#c7e9b4"
        if year == "0003" and not sub.empty:
            color = "#fde0dd"
        ax.add_patch(plt.Rectangle((mi, yi), 1, 1, facecolor=color, edgecolor="0.35", lw=0.8))
        if sub.empty:
            text = "missing"
        else:
            lines = []
            for _, row in sub.iterrows():
                tag = row["config"]
                if row["perturbation"] != "small_temperature":
                    tag += " " + row["perturbation"].split("_")[0]
                lines.append(f"{tag} n={int(row['n_members'])}")
            if year == "0003":
                lines.append("special branch")
            text = "\n".join(lines)
        ax.text(mi + 0.5, yi + 0.5, text, ha="center", va="center", fontsize=8)
ax.set_title("Hindcast availability matrix")
ax.set_xlabel("Initialization month")
ax.set_ylabel("Reference year")
savefig(fig, "00_hindcast_availability_matrix", notebook="00_case_inventory_and_utils.ipynb", scientific_question="Which hindcast cases are available for source and feedback diagnostics?", variables_windows="case inventory; products partial_O3, EPflux_daily_ubar, U, Z3, FWD, AO/NAM", interpretation="Green paired cells support INT-vs-NOCOUPL tests; red/pink 0003 cells are special-branch INT-only caveats.", caveat="Availability is product-level; individual member gaps are logged during later diagnostics.", csv_table=inv_path)
plt.show()
write_figure_guide()